# Title
Cutting Edge Challengers
## Purpose
Document how optional TabPFN and AutoGluon challengers plug into the benchmark runner.
## Inputs
`data/gold/listings_gold.xlsx`, the challenger helpers, and the advanced dependency set.
## Outputs
Optional challenger metrics, predictions, and external artifacts under `results/benchmarks/...`.
## Key Takeaways
This notebook is intentionally opt-in because the advanced dependencies are heavier than the default stack.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

def resolve_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'config.py').exists():
            return candidate
    raise FileNotFoundError('Could not find config.py in the current path ancestry')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import LISTINGS_GOLD, RESULTS_DIR
from elferspot_listings.modeling.challengers import run_autogluon_regression, run_tabpfn_regression
from elferspot_listings.modeling.train import train_baseline_models

BENCHMARK_DIR = RESULTS_DIR / 'benchmarks' / '03_cutting_edge_challengers'
RUN_CHALLENGERS = False
RUN_TRAINING = False
ADVANCED_DEPENDENCIES_NOTE = 'Install the optional stack with `python -m pip install -r requirements-advanced.txt`.'


In [ ]:
gold_exists = LISTINGS_GOLD.exists()
metrics_path = BENCHMARK_DIR / 'metrics.json'
predictions_path = BENCHMARK_DIR / 'predictions.csv'

print(f'Gold input: {LISTINGS_GOLD}')
print(f'Gold present: {gold_exists}')
print(f'Benchmark directory: {BENCHMARK_DIR}')
print(ADVANCED_DEPENDENCIES_NOTE)

if gold_exists:
    gold_preview = pd.read_excel(LISTINGS_GOLD, nrows=3)
    print('\nGold preview:')
    print(gold_preview.to_string(index=False))


In [ ]:
if RUN_TRAINING and gold_exists:
    gold_df = pd.read_excel(LISTINGS_GOLD)
    result = train_baseline_models(
        gold_df,
        BENCHMARK_DIR,
        run_tabpfn=RUN_CHALLENGERS,
        run_autogluon=RUN_CHALLENGERS,
    )
    print(f'Trained models: {sorted(result.metrics)}')
    print('Challengers are enabled through train_baseline_models(..., run_tabpfn=True, run_autogluon=True).')
elif metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Loaded existing benchmark metrics:')
    print(pd.DataFrame(metrics).T.sort_values('mae_eur').to_string())
    if predictions_path.exists():
        predictions = pd.read_csv(predictions_path)
        challenger_rows = predictions[predictions['model_name'].isin(['tabpfn', 'autogluon'])]
        print('\nChallenger prediction sample:')
        print(challenger_rows.head().to_string(index=False))
else:
    print('No challenger outputs found yet. Keep RUN_CHALLENGERS=False for review-only use, or enable it when advanced dependencies are installed.')
